In [ ]:
import numpy as np
import pandas as pd
import math

data = pd.DataFrame({
    "Wind": ["Weak", "Strong", "Weak", "Weak", "Strong"],
    "Humidity": ["High", "High", "Low", "Low", "Low"],
    "Rain": ["Yes", "Yes", "No", "Yes", "No"]
})
data

,Wind,Humidity,Rain
0,Weak,High,Yes
1,Strong,High,Yes
2,Weak,Low,No
3,Weak,Low,Yes
4,Strong,Low,No


In [ ]:
classes = ["Yes", "No"]
feature_values = {
    "Wind": ["Weak", "Strong"],
    "Humidity": ["High", "Low"]
}

priors = {c: (data["Rain"] == c).mean() for c in classes}

def laplace_prob(feature, value, cls, alpha=1):
    subset = data[data["Rain"] == cls]
    count = (subset[feature] == value).sum()
    k = len(feature_values[feature])
    return (count + alpha) / (len(subset) + alpha * k)

likelihood_rows = []
for c in classes:
    for feature in ["Wind", "Humidity"]:
        for value in feature_values[feature]:
            likelihood_rows.append([
                c, feature, value, laplace_prob(feature, value, c)
            ])

print("Class priors:", priors)
pd.DataFrame(
    likelihood_rows,
    columns=["Class", "Feature", "Value", "Laplace-smoothed probability"]
)


Class priors: {'Yes': np.float64(0.6), 'No': np.float64(0.4)}


,Class,Feature,Value,Laplace-smoothed probability
0,Yes,Wind,Weak,0.60
1,Yes,Wind,Strong,0.40
2,Yes,Humidity,High,0.60
3,Yes,Humidity,Low,0.40
4,No,Wind,Weak,0.50
5,No,Wind,Strong,0.50
6,No,Humidity,High,0.25
7,No,Humidity,Low,0.75


In [ ]:
conditions = [
    ("Weak", "High"),
    ("Strong", "High"),
    ("Weak", "Low"),
    ("Strong", "Low")
]

rows = []
for wind, humidity in conditions:
    scores = {}
    for c in classes:
        scores[c] = (
            priors[c]
            * laplace_prob("Wind", wind, c)
            * laplace_prob("Humidity", humidity, c)
        )

    normalizer = scores["Yes"] + scores["No"]
    p_yes = scores["Yes"] / normalizer
    p_no = scores["No"] / normalizer
    prediction = "Yes" if p_yes > p_no else "No"
    action = "Do not deliver" if prediction == "Yes" else "Deliver"

    rows.append([
        wind, humidity,
        scores["Yes"], scores["No"],
        p_yes, p_no, prediction, action
    ])

nb_results = pd.DataFrame(rows, columns=[
    "Wind", "Humidity",
    "Unnorm Yes", "Unnorm No",
    "P(Yes | evidence)", "P(No | evidence)",
    "Prediction", "Drone action"
])

nb_results.round(4)


,Wind,Humidity,Unnorm Yes,Unnorm No,P(Yes | evidence),P(No | evidence),Prediction,Drone action
0,Weak,High,0.216,0.05,0.8120,0.1880,Yes,Do not deliver
1,Strong,High,0.144,0.05,0.7423,0.2577,Yes,Do not deliver
2,Weak,Low,0.144,0.15,0.4898,0.5102,No,Deliver
3,Strong,Low,0.096,0.15,0.3902,0.6098,No,Deliver


In [ ]:
def entropy(labels):
    counts = pd.Series(labels).value_counts()
    probs = counts / counts.sum()
    return -sum(p * math.log2(p) for p in probs)

def information_gain(df, feature, target="Rain"):
    parent_entropy = entropy(df[target])
    remainder = 0.0
    for _, subset in df.groupby(feature):
        remainder += len(subset) / len(df) * entropy(subset[target])
    return parent_entropy, remainder, parent_entropy - remainder

H_rain = entropy(data["Rain"])
wind_stats = information_gain(data, "Wind")
humidity_stats = information_gain(data, "Humidity")

print(f"H(Rain) = {H_rain:.6f}")
print(f"Wind: weighted entropy = {wind_stats[1]:.6f}, IG = {wind_stats[2]:.6f}")
print(f"Humidity: weighted entropy = {humidity_stats[1]:.6f}, IG = {humidity_stats[2]:.6f}")


H(Rain) = 0.970951
Wind: weighted entropy = 0.950978, IG = 0.019973
Humidity: weighted entropy = 0.550978, IG = 0.419973


In [ ]:
low = data[data["Humidity"] == "Low"]
H_low = entropy(low["Rain"])
_, rem_low_wind, ig_low_wind = information_gain(low, "Wind")

print(f"Entropy in Humidity=Low subset = {H_low:.6f}")
print(f"IG of Wind within Humidity=Low subset = {ig_low_wind:.6f}")


Entropy in Humidity=Low subset = 0.918296
IG of Wind within Humidity=Low subset = 0.251629


In [ ]:
from scipy.optimize import minimize

X = np.array([
    [0, 1],  # Weak, High
    [1, 1],  # Strong, High
    [0, 0],  # Weak, Low
    [0, 0],  # Weak, Low
    [1, 0],  # Strong, Low
], dtype=float)

y = np.array([1, 1, 0, 1, 0], dtype=float)

Xb = np.column_stack([np.ones(len(X)), X])
lam = 0.1

def regularized_log_loss(beta):
    z = Xb @ beta
    data_loss = np.sum(np.logaddexp(0, z) - y * z)
    penalty = (lam / 2) * np.sum(beta[1:] ** 2)  # intercept not penalized
    return data_loss + penalty

result = minimize(regularized_log_loss, np.zeros(3), method="BFGS")
beta = result.x

print("beta0 (intercept)      =", round(beta[0], 6))
print("beta1 (Wind_Strong)    =", round(beta[1], 6))
print("beta2 (Humidity_High)  =", round(beta[2], 6))


beta0 (intercept)      = -0.037122
beta1 (Wind_Strong)    = -0.950323
beta2 (Humidity_High)  = 2.528478


In [ ]:
test_X = np.array([
    [0, 1],  # Weak, High
    [1, 1],  # Strong, High
    [0, 0],  # Weak, Low
    [1, 0],  # Strong, Low
], dtype=float)

test_names = [
    ("Weak", "High"),
    ("Strong", "High"),
    ("Weak", "Low"),
    ("Strong", "Low"),
]

test_Xb = np.column_stack([np.ones(len(test_X)), test_X])
z = test_Xb @ beta
p_yes = 1 / (1 + np.exp(-z))

lr_rows = []
for (wind, humidity), prob in zip(test_names, p_yes):
    prediction = "Yes" if prob >= 0.5 else "No"
    action = "Do not deliver" if prediction == "Yes" else "Deliver"
    lr_rows.append([wind, humidity, prob, prediction, action])

lr_results = pd.DataFrame(
    lr_rows,
    columns=["Wind", "Humidity", "P(Rain=Yes)", "Prediction", "Drone action"]
)
lr_results.round(4)


,Wind,Humidity,P(Rain=Yes),Prediction,Drone action
0,Weak,High,0.9235,Yes,Do not deliver
1,Strong,High,0.8236,Yes,Do not deliver
2,Weak,Low,0.4907,No,Deliver
3,Strong,Low,0.2714,No,Deliver
